# 03 · Modelagem experimental e validação fora do tempo

**Objetivo operacional hipotético:** estimar no momento M0 a chance do evento composto `RETIDO` com alguma movimentação até o encerramento do funil, **entre registros com desfecho encerrado**. Trata-se de classificação experimental, não do efeito causal de oferta.

**Separação:** treino junho–setembro de 2025, validação outubro, teste novembro–dezembro. Clientes presentes em período posterior são removidos dos períodos anteriores para reduzir vazamento entre cortes. Registros repetidos dentro do mesmo período permanecem; o horizonte do rótulo não é datado.

**Controle:** baseline constante, regressão logística e árvore curta. Escolha por Brier score da validação, teste reservado ao final. Reportam-se AUC-ROC, AP, Brier, log-loss e lift nos 10% de maior escore. Sem calibração pós-hoc nem tuning em teste.

**Antes de reproduzir:** executar `python -m retention_analytics.analysis_cli --input data/raw/retention_case.xlsx --out-dir reports/private`. Não executar em repositório público com dados reais.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / 'pyproject.toml').exists():
    root = root.parent
private = root / 'reports' / 'private'

metrics = json.loads((private / 'model_validation.json').read_text(encoding='utf-8'))
print('Alvo:', metrics['task'])
print('Modelo selecionado:', metrics['selected_by_validation_brier'])
print('Partições:', metrics['split_counts'])

## Resultados da validação (escolha de modelo)

A interpretação de AUC/AP deve considerar prevalência e seleção de casos encerrados. O modelo não foi avaliado em período posterior a dezembro de 2025.

In [ ]:
for model_name, data in metrics['validation_metrics'].items():
    print(model_name, {k: round(v, 4) for k, v in data.items()
                       if k not in ('n',)})

## Holdout cronológico: avaliação final

A linha de base usa apenas a prevalência do treino. O modelo escolhido não é automaticamente aceitável para implementação operacional; avaliar estabilidade, vieses geográficos, deriva e calibração em novos dados.

In [ ]:
for model_name, data in metrics['test_metrics'].items():
    print(model_name, {k: round(v, 4) for k, v in data.items()
                       if k != 'n'})
names = list(metrics['test_metrics'])
auc = [metrics['test_metrics'][x]['roc_auc'] for x in names]
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.bar(names, auc)
ax.axhline(0.5, color='grey', linestyle='--')
ax.set_ylim(0, 1)
ax.set(ylabel='AUC-ROC no teste', title='Discriminação no período de teste')
plt.show()

## Restrições para uso de negócio

O desfecho é provisório e medido após contato. Leads abertos são excluídos e podem gerar viés de maturidade. O reuso de clientes foi mitigado entre splits, mas não dentro deles. Coeficientes ou importância de atributos não são efeitos causais. Antes de qualquer uso como recomendador é necessária validação temporal prospectiva, auditoria de justiça/risco de proxy, calibração e teste controlado de ofertas.